# Sentiment labels per sentence, saved for the topic notebook

Willian Angelo · Team 10 · Bank of England employer project (CAM_DS_401) · release `2026-09-30`

**Where the labels come from.** The team's sentiment notebook, `notebooks/rachel/All_in_one/sentiment_classification.ipynb` (branch `rachel`, commit `0c39267` of 5 October 2026, written by Rachel Robinson), labels every sentence with four models: an emotion model (BERT-emotion), FinBERT, FinBERT-tone and a RoBERTa sentiment model. It keeps the labels only in memory, summarises them by quarter, and runs one bank at a time, so no file of labels exists.

**What this notebook does.** It copies that notebook's labelling cells unchanged, runs them on both banks, and saves the labels as one row per sentence with the key the topic notebook (`angelo_a3.ipynb`) joins on. The labels are the sentiment track's own: same input file, same cleaning, same models, same settings. Every copied cell says which cell of the source notebook it comes from. Three things differ, and none of them changes a label:

1. **Loading.** The data comes through the team loader from release `2026-09-30`. The source notebook reads `revised_structured_sentences.csv` from a folder on its author's laptop; that file is identical to the release's `structured_sentences.csv` (same 19,092 rows and values, checked on 5 October).
2. **Both banks in one run.** The source notebook keeps one bank before running the models (its cell 17). Each sentence is labelled on its own, and the list of speaker names used in the cleaning is built before that filter, so a sentence gets the same label either way. The bank filter is therefore left out.
3. **GPU.** `device=DEVICE` is added to the four `pipeline(...)` calls, so the models run on the GPU when there is one. Step 4.1 compares the result with the labels saved in the source notebook's outputs.

The saved file holds keys and labels only, no sentence text. The outputs of this notebook print transcript text: **clear all outputs before every commit** (public repository).

## Environment

Team setup cell with `NAME = "angelo"`, then `EXPORT = "2026-09-30"`. The cells below expect `ROOT`, `DATA`, `EXPORT` and `loading`.

In [ ]:
# 0.0 local Windows only, run before the setup cell: pandas loads pyarrow, whose bundled msvcp140.dll
# is older than the one torch needs, and torch then fails with WinError 1114. Harmless in Colab.
import torch

In [ ]:
# project setup - run first, do not edit except NAME
NAME = "angelo"  # <<< your name

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
               check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}  data {DATA}")

In [ ]:
EXPORT = "2026-09-30"  # pin the export this notebook was written against
str_sent = loading.load(DATA, "structured_sentences.csv", EXPORT)

## Settings and data

In [ ]:
# 1.0 settings: output folder, device, and the key shared with the topic notebook
try:
    import torch
except OSError as e:
    raise OSError("torch failed to load because pandas loaded first: restart the kernel "
                  "and run the 0.0 cell before the setup cell") from e
import time
import numpy as np
import pandas as pd

for name in ("ROOT", "DATA", "EXPORT", "loading"):
    if name not in globals():
        raise NameError(f"run the Environment cell first: {name} is not defined")
OUT = ROOT / "notebooks" / "angelo" / "outputs" / EXPORT      # outputs/ is kept out of git by .git/info/exclude
if DATA in OUT.parents:
    raise RuntimeError(f"never write inside {DATA}: it holds the read-only release")
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"        # added to the source notebook's pipeline calls
KEY = ["bank", "quarter", "call_type", "position_in_call", "sentence_number"]
pd.set_option("display.max_colwidth", 100)
print(f"release {EXPORT}   device {DEVICE}   outputs {OUT}")

In [ ]:
# 1.1 the source notebook's cell 5, with the release in place of the file on its author's laptop (identical content)
data = str_sent.copy()
print(data.shape)

## Cleaning, copied from the sentiment notebook

These five cells are the source notebook's cells 6, 9, 11, 13 and 14, unchanged. They drop filler sentences, then turn each sentence into a list of words: lower case, letters only, no English stop words, no speaker names, each word reduced to its dictionary form. The models in the next section read these word lists, joined back into a string, not the original sentence.

In [ ]:
# copied from the source notebook, cell 6 (unchanged)
# Removerows where is_filler_sentence = True
data = data[data['is_filler_sentence'] != True]
print(data.shape)

In [ ]:
# copied from the source notebook, cell 9 (unchanged)
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
# copied from the source notebook, cell 11 (unchanged)
# Create a list using "speaker_name" column from the dataframe
speaker_names = data["speaker_name"].tolist()
# Reduce list to contain only unique speaker names
speaker_names = list(set(speaker_names))
# Split first and last names of the speakers
speaker_names_split = [name.split() for name in speaker_names]
# remove capital letters from speaker names
speaker_names_split = [[name.lower() for name in sublist] for sublist in speaker_names_split]
speaker_names_split[:10]

In [ ]:
# copied from the source notebook, cell 13 (unchanged)
def preprocess_text(text):
    if not isinstance(text, str): # Handle non-string input, e.g., NaN
        return []
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize the text
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize
    stop_words = set(stopwords.words('english'))

    # Remove speaker names from the text
    speaker_names_flat = [name for sublist in speaker_names_split for name in sublist]

    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        # Removed "greeting words" from the line below:
        if word not in stop_words and word.isalpha() and word and word not in speaker_names_flat # Ensure only alphabetic words are kept and remove greeting words and speaker names
    ]
    return [word for word in cleaned_tokens if word not in stop_words]

In [ ]:
# copied from the source notebook, cell 14 (unchanged), then the shape its cell 15 shows: (16513, 16)
# Apply the preprocessing function
data['cleaned_text'] = data['sentence'].apply(preprocess_text)

# Display the first few rows
print(data[['sentence', 'cleaned_text']].head(10))
print(data.shape)

## The four models, copied from the sentiment notebook

Each model has two cells, as in the source notebook: one loads the model and builds the list of texts, one runs it and adds the labels to `data`. The only change is `device=DEVICE` in the `pipeline(...)` call; a timing line is added at the start and end of each run. The source notebook's comments still say "UBS" because it ran one bank at a time; here every sentence of both banks is labelled.

| Model | Hugging Face name | Columns added to `data` |
|---|---|---|
| BERT-emotion | `bhadresh-savani/bert-base-uncased-emotion` | `main_emotion`, `main_emotion_score`, one score per emotion (`anger`, `joy`, `fear`, `sadness`, `love`, `surprise`) |
| FinBERT | `prosusai/finbert` | `main_sentiment`, `main_sentiment_score`, `positive`, `negative`, `neutral` |
| FinBERT-tone | `yiyanghkust/finbert-tone` | `FBT_Main_Sentiment`, `FBT_Main_Sentiment_Score`, `FBT_Positive`, `FBT_Negative`, `FBT_Neutral` |
| RoBERTa | `cardiffnlp/twitter-roberta-base-sentiment` | `RBT_main_sentiment`, `RBT_main_sentiment_score`, `RBT_negative`, `RBT_neutral`, `RBT_positive` |

In [ ]:
# copied from the source notebook, cell 26; only change: device=DEVICE
# Conduct Bert Emotion analysis on cleaned sentences from presentation section (UBS)
from transformers import pipeline
emotion_analyzer = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion", device=DEVICE)
documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

In [ ]:
# copied from the source notebook, cell 28 (unchanged apart from the timing lines)
t0 = time.time()  # added: timing
# Run the model and keep all emotion scores
basic_emotion = emotion_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
emotion_rows = []
for result in basic_emotion:
    if isinstance(result, list):
        emotion_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        emotion_rows.append({result["label"]: result["score"]})
    else:
        emotion_rows.append({})

# Add the top emotion and score to the dataframe
data["main_emotion"] = [
    max(row, key=row.get, default="neutral") if row else "neutral"
    for row in emotion_rows
]
data["main_emotion_score"] = [
    row.get(max(row, key=row.get, default="neutral"), 0.0) if row else 0.0
    for row in emotion_rows
]

# Add all emotion columns as separate numeric columns
for emotion in ["anger", "joy", "fear", "sadness", "love", "surprise"]:
    data[emotion] = [row.get(emotion, 0.0) for row in emotion_rows]
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

In [ ]:
# copied from the source notebook, cell 43; only change: device=DEVICE
# Conduct sentiment analysis using FinBERT on UBS-related financial texts
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

# Load FinBERT model and tokenizer
model_name = "prosusai/finbert"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=DEVICE)

documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

In [ ]:
# copied from the source notebook, cell 45 (unchanged apart from the timing lines)
t0 = time.time()  # added: timing
# Run the model and keep all sentiment scores
basic_sentiment = sentiment_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
sentiment_rows = []
for result in basic_sentiment:
    if isinstance(result, list):
        sentiment_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        sentiment_rows.append({result["label"]: result["score"]})
    else:
        sentiment_rows.append({})

# Add the top sentiment and score to the dataframe
data["main_sentiment"] = [
    max(row, key=row.get, default="neutral") if row else "neutral"
    for row in sentiment_rows
]
data["main_sentiment_score"] = [
    row.get(max(row, key=row.get, default="neutral"), 0.0) if row else 0.0
    for row in sentiment_rows
]

# Add all sentiment columns as separate numeric columns
for sentiment in ["positive", "negative", "neutral"]:
    data[sentiment] = [row.get(sentiment, 0.0) for row in sentiment_rows]
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

In [ ]:
# copied from the source notebook, cell 61; only change: device=DEVICE
# Conduct sentiment analysis using FinBERT on UBS-related financial texts
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

# Load FinBERT model and tokenizer
model_name = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=DEVICE)

documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

In [ ]:
# copied from the source notebook, cell 63 (unchanged apart from the timing lines), then its cell 65, which
# renames the FinBERT-tone score columns (its trailing data.head(10) left out)
t0 = time.time()  # added: timing
# Run the model and keep all sentiment scores
basic_FBT_sentiment = sentiment_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
sentiment_rows = []
for result in basic_FBT_sentiment:
    if isinstance(result, list):
        sentiment_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        sentiment_rows.append({result["label"]: result["score"]})
    else:
        sentiment_rows.append({})

# Add the top sentiment and score to the dataframe
data["FBT_Main_Sentiment"] = [
    max(row, key=row.get, default="Neutral") if row else "Neutral"
    for row in sentiment_rows
]
data["FBT_Main_Sentiment_Score"] = [
    row.get(max(row, key=row.get, default="Neutral"), 0.0) if row else 0.0
    for row in sentiment_rows
]

# Add all sentiment columns as separate numeric columns
# NOTE: FinBERT-Tone requires capital letters for the sentiment labels
for sentiment in ["Positive", "Negative", "Neutral"]:
    data[sentiment] = [row.get(sentiment, 0.0) for row in sentiment_rows]
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

# Update FinBERT-Tone column names to make it clearer
data.rename(
    columns={
        "Positive": "FBT_Positive",
        "Negative": "FBT_Negative",
        "Neutral": "FBT_Neutral",
    },
    inplace=True,
)

In [ ]:
# copied from the source notebook, cell 80; only change: device=DEVICE
# Conduct sentiment analysis using FinBERT on UBS-related financial texts
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load RoBERTa model and tokenizer
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=DEVICE)

documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

In [ ]:
# copied from the source notebook, cell 82 (unchanged apart from the timing lines)
t0 = time.time()  # added: timing
# Run the model and keep all sentiment scores
basic_sentiment = sentiment_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
sentiment_rows = []
for result in basic_sentiment:
    if isinstance(result, list):
        sentiment_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        sentiment_rows.append({result["label"]: result["score"]})
    else:
        sentiment_rows.append({})

# Add the top sentiment and score to the dataframe
data["RBT_main_sentiment"] = [
    max(row, key=row.get, default="LABEL_1") if row else "LABEL_1"
    for row in sentiment_rows
]
data["RBT_main_sentiment_score"] = [
    row.get(max(row, key=row.get, default="LABEL_1"), 0.0) if row else 0.0
    for row in sentiment_rows
]

# Add all sentiment columns as separate numeric columns
for sentiment in ["LABEL_0", "LABEL_1", "LABEL_2"]:
    data[sentiment] = [row.get(sentiment, 0.0) for row in sentiment_rows]
data.rename(columns={"LABEL_0": "RBT_negative", "LABEL_1": "RBT_neutral", "LABEL_2": "RBT_positive"}, inplace=True)

# Map main_sentiment to "LABEL_0": "negative","LABEL_1": "neutral" and "LABEL_2": "positive"
label_mapping = {
    "LABEL_0": "RBT_negative",
    "LABEL_1": "RBT_neutral",
    "LABEL_2": "RBT_positive",
}
data["RBT_main_sentiment"] = data["RBT_main_sentiment"].map(label_mapping)
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

## Check and save

In [ ]:
# 4.1 does this run give the source notebook's labels? Its saved outputs show the labels of the first UBS
# sentences (cells 29, 46, 64 and 83, scores rounded to 6 decimals). Labels must match exactly, scores to 1e-4.
REFERENCE = {   # sentence_id: label and score per model, read from the source notebook's saved outputs
    33595: {"main_emotion": ("joy", 0.970431), "main_sentiment": ("neutral", 0.876172),
            "FBT_Main_Sentiment": ("Neutral", 0.633788), "RBT_main_sentiment": ("RBT_neutral", 0.585431)},
    33597: {"main_emotion": ("anger", 0.889760), "main_sentiment": ("negative", 0.720507),
            "FBT_Main_Sentiment": ("Neutral", 0.762569), "RBT_main_sentiment": ("RBT_neutral", 0.798390)},
    33598: {"main_sentiment": ("neutral", 0.920784), "FBT_Main_Sentiment": ("Neutral", 0.995665),
            "RBT_main_sentiment": ("RBT_neutral", 0.609302)},
    33599: {"main_sentiment": ("positive", 0.841943), "FBT_Main_Sentiment": ("Positive", 1.000000),
            "RBT_main_sentiment": ("RBT_positive", 0.747497)},
    33600: {"main_sentiment": ("neutral", 0.836954), "FBT_Main_Sentiment": ("Positive", 0.999991),
            "RBT_main_sentiment": ("RBT_positive", 0.880521)},
    33601: {"main_sentiment": ("neutral", 0.875320), "FBT_Main_Sentiment": ("Neutral", 0.770857),
            "RBT_main_sentiment": ("RBT_neutral", 0.771707)},
    33602: {"main_sentiment": ("neutral", 0.877357), "FBT_Main_Sentiment": ("Positive", 0.493373),
            "RBT_main_sentiment": ("RBT_neutral", 0.819947)},
    33603: {"main_sentiment": ("positive", 0.539305), "FBT_Main_Sentiment": ("Positive", 0.892135),
            "RBT_main_sentiment": ("RBT_neutral", 0.539774)},
    33604: {"main_sentiment": ("positive", 0.953584), "FBT_Main_Sentiment": ("Positive", 1.000000),
            "RBT_main_sentiment": ("RBT_positive", 0.834321)},
    33605: {"main_sentiment": ("neutral", 0.763131), "FBT_Main_Sentiment": ("Positive", 0.962873),
            "RBT_main_sentiment": ("RBT_positive", 0.789465)},
}
SCORE = {"main_emotion": "main_emotion_score", "main_sentiment": "main_sentiment_score",
         "FBT_Main_Sentiment": "FBT_Main_Sentiment_Score", "RBT_main_sentiment": "RBT_main_sentiment_score"}
by_id = data.set_index("sentence_id")
rows = []
for sid, models in REFERENCE.items():
    for col, (label, score) in models.items():
        rows.append({"sentence_id": sid, "model column": col, "source label": label, "this run": by_id.at[sid, col],
                     "score difference": abs(by_id.at[sid, SCORE[col]] - score)})
check = pd.DataFrame(rows)
check["same label"] = check["source label"] == check["this run"]
display(check.groupby("model column").agg(compared=("same label", "size"), same_label=("same label", "sum"),
                                          largest_score_difference=("score difference", "max")))
assert check["same label"].all() and (check["score difference"] < 1e-4).all(), "this run differs from the source"
print(f"all {len(check)} saved labels reproduced")

In [ ]:
# 4.2 what the labels say: share of each label per model, by bank and section (% of sentences), and how often
# the three sentiment models agree that a sentence is negative
LABEL_OF = {"BERT-emotion": "main_emotion", "FinBERT": "main_sentiment", "FinBERT-tone": "FBT_Main_Sentiment",
            "RoBERTa": "RBT_main_sentiment"}
for name, col in LABEL_OF.items():
    print(f"\n{name}: % of sentences per label")
    display((100 * pd.crosstab([data["bank"], data["section"]], data[col], normalize="index")).round(1))

negative = pd.DataFrame({"FinBERT": data["main_sentiment"] == "negative",
                         "FinBERT-tone": data["FBT_Main_Sentiment"] == "Negative",
                         "RoBERTa": data["RBT_main_sentiment"] == "RBT_negative"})
votes = negative.sum(axis=1)
print("sentences called negative by 0, 1, 2 or 3 of the sentiment models:", votes.value_counts().sort_index().to_dict())
print("pairwise agreement on 'negative or not' (%):",
      {f"{a} / {b}": round(100 * (negative[a] == negative[b]).mean(), 1)
       for i, a in enumerate(negative) for b in list(negative)[i + 1:]})

In [ ]:
# 4.3 save one row per sentence: the key, the sentence id, section and role, and every model's label and scores.
# No sentence text, so the file holds no transcript content.
LABEL_COLS = ["main_emotion", "main_emotion_score", "anger", "joy", "fear", "sadness", "love", "surprise",
              "main_sentiment", "main_sentiment_score", "positive", "negative", "neutral",
              "FBT_Main_Sentiment", "FBT_Main_Sentiment_Score", "FBT_Positive", "FBT_Negative", "FBT_Neutral",
              "RBT_main_sentiment", "RBT_main_sentiment_score", "RBT_negative", "RBT_neutral", "RBT_positive"]
labels = data[KEY + ["sentence_id", "section", "speaker_role"] + LABEL_COLS]
assert not labels.duplicated(KEY).any(), "the key must identify one sentence"
labels.to_csv(OUT / "sentiment_labels_sentence.csv", index=False)
print(f"wrote sentiment_labels_sentence.csv: {len(labels):,} sentences x {labels.shape[1]} columns")
display(labels.groupby(["bank", "section"]).size().unstack())

### Summary

(written after the run)